# Trabajo Práctico Integrador: Análisis de Desempeño y Gestión de Estudiantes 🎓
**Materia:** Análisis de Datos Inicial  
**Carrera:** Tecnicatura Universitaria en Programación (TUP)

**Integrantes del grupo:**
Gonzalo Betancourt

---

## Hito 1: Elección y Planteo 🎯

**Dataset elegido:** OULAD (Open University Learning Analytics Dataset). Trabajaremos con una base de datos relacional cruzando la información demográfica (`studentInfo.csv`), el registro de entregas (`studentAssessment.csv`) y los detalles de las evaluaciones (`assessments.csv`).

**Objetivos del Análisis (Preguntas a responder):**
1. **Métrica de Engagment:** ¿Existe un umbral de interacciones (entregas tempranas y participación) durante las primeras semanas que actúe como predictor estadístico del estado final de abandono ('Withdrawn')?
2. **Rendimiento vs. Evaluación:** ¿Cómo varía la distribución de calificaciones y la varianza del desempeño dependiendo de la tipología de la evaluación (TMA, CMA, Exam)?
3. **Análisis Demográfico:** ¿Qué impacto probabilístico tienen las variables de índice de privación (IMD Band) y el nivel educativo previo sobre la tasa de aprobación final?


In [ ]:
# Importación de librerías necesarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual básica para los gráficos
sns.set_theme(style="whitegrid")

---
## Hito 2: Limpieza y Preparación 🧹

En esta sección vamos a cargar los datos y realizar una auditoría inicial para entender con qué estamos trabajando. Luego, aplicaremos las técnicas de limpieza necesarias.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carga de datos
try:
    student_info = pd.read_csv('studentInfo.csv')
    student_assessment = pd.read_csv('studentAssessment.csv')
    assessments = pd.read_csv('assessments.csv')
    print("Datos cargados correctamente.")
except FileNotFoundError:
    print("Error: Asegúrate de tener los archivos CSV en el mismo directorio.")

# 2. Merge Eficiente de Tablas
df_eval = pd.merge(student_assessment, assessments, on='id_assessment', how='inner')
df_master = pd.merge(df_eval, student_info, on=['id_student', 'code_module', 'code_presentation'], how='inner')

print("\n--- Primeras 5 filas del dataset ---")
display(df_master.head())

print("\n--- Información del dataset ---")
df_master.info()


### Aplicación de Limpieza (Data Cleaning)
A continuación, procesaremos los datos nulos, corregiremos tipos de datos (por ejemplo, si las notas se leen como texto en lugar de números) y renombraremos columnas si es necesario.

In [ ]:
# 3. Tratamiento Avanzado de Nulos (Imputación Estadística)
df_master['score'] = df_master.groupby('assessment_type')['score'].transform(lambda x: x.fillna(x.median()))

moda_imd = df_master['imd_band'].mode()[0]
df_master['imd_band'] = df_master['imd_band'].fillna(moda_imd)

# 4. Eliminación de Outliers mediante Rango Intercuartílico (IQR) para Calificaciones
Q1 = df_master['score'].quantile(0.25)
Q3 = df_master['score'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

df_clean = df_master[(df_master['score'] >= limite_inferior) & (df_master['score'] <= limite_superior)].copy()

# 5. Feature Engineering
df_clean['weight_adj'] = df_clean['weight'].replace(0, 1)
df_clean['Indice_Rendimiento'] = df_clean.groupby('code_module')['score'].transform(lambda x: (x - x.mean()) / x.std())
df_clean['Indice_Rendimiento'] = df_clean['Indice_Rendimiento'].fillna(0)


# 6. Cálculo de Retraso (Feature Engineering para Pregunta 1)
df_clean['date'] = pd.to_numeric(df_clean['date'], errors='coerce')
df_clean['retraso'] = df_clean['date_submitted'] - df_clean['date']
df_clean['retraso'] = df_clean['retraso'].fillna(0)
print(f"Filas originales: {len(df_master)} | Filas tras remover outliers: {len(df_clean)}")


---
## Hito 3: Análisis y Visualización 📊

Es momento de explorar los datos visualmente y responder a las preguntas planteadas en el Hito 1.

In [ ]:
df_clean.describe()

### Visualización 1: Distribución (Ej: Histograma de Calificaciones)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

plt.figure(figsize=(10, 6))
sns.violinplot(data=df_clean, x='assessment_type', y='score', hue='assessment_type', palette='Set2', legend=False)
plt.title('Distribución de Calificaciones según Tipo de Evaluación', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de Evaluación')
plt.ylabel('Calificación (Score)')
plt.show()


**Conclusión de la Visualización 1:** El gráfico de violín revela que los exámenes tradicionales (Exam) presentan una distribución más compacta y centrada, mientras que las evaluaciones continuas (TMA/CMA) muestran mayor varianza, sugiriendo que la evaluación a lo largo del curso genera una dispersión más heterogénea en el rendimiento estudiantil.


### Visualización 2 y 3: Relación y Categorías
*(Utilicen celdas de código para generar gráficos de barras, scatterplots u otros que respondan a sus preguntas iniciales. No olviden poner títulos y etiquetas a los ejes).*

In [ ]:
# --- GRÁFICO 1: RETRASO EN ENTREGAS VS ABANDONO (Responde Pregunta 1) ---
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_clean, x='final_result', y='retraso', palette='Set2')
plt.title('Distribución de Retraso en Entregas según Estado Final', fontsize=14, fontweight='bold')
plt.xlabel('Estado Final (Pass, Fail, Withdrawn)')
plt.ylabel('Días de Retraso (Positivo = Tarde, Negativo = Temprano)')
plt.ylim(-50, 50) # Limitamos a 50 días para mejor visualización y no ver outliers extremos
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Fecha Límite')
plt.legend()
plt.show()

# --- GRÁFICO 2: IMPACTO DEMOGRÁFICO (EDUCACIÓN PREVIA) ---
plt.figure(figsize=(10, 6))
df_clean['Aprobado'] = df_clean['final_result'].isin(['Pass', 'Distinction']).astype(int)
educ_pass_rate = df_clean.groupby('highest_education')['Aprobado'].mean().sort_values(ascending=False) * 100

sns.barplot(x=educ_pass_rate.values, y=educ_pass_rate.index, hue=educ_pass_rate.index, palette='viridis', legend=False)
plt.title('Tasa de Aprobación por Nivel Educativo Previo', fontsize=14, fontweight='bold')
plt.xlabel('Tasa de Aprobación (%)')
plt.ylabel('Nivel Educativo')
plt.show()

# --- GRÁFICO 3: ÍNDICE DE PRIVACIÓN VS RESULTADOS ---
plt.figure(figsize=(12, 6))
imd_order = sorted(df_clean['imd_band'].unique())
sns.countplot(data=df_clean, x='imd_band', hue='final_result', order=imd_order, palette='coolwarm')
plt.title('Resultados Finales según Índice de Privación (IMD Band)', fontsize=14, fontweight='bold')
plt.xlabel('Índice de Privación (0-10% es la zona más carenciada)')
plt.ylabel('Cantidad de Evaluaciones')
plt.xticks(rotation=45)
plt.legend(title='Resultado Final')
plt.show()


## Hito 4: Interfaz Gráfica y Dashboard Interactivo 🖥️

Como estudiantes de programación, en esta etapa deben trascender el análisis estático. El objetivo es construir una herramienta funcional que permita a un usuario (gestor educativo, docente o director) interactuar con los datos sin necesidad de ver el código.

### 🔗 Entregables del Hito
* **Repositorio de GitHub:** https://github.com/Gonzalo-Betancourt/Parical1_Analisis_Datos

### 🛠️ Especificaciones Técnicas
Para la aprobación de este hito, la interfaz debe cumplir con:
1.  **Interactividad:** Uso de widgets (selectboxes, sliders, multiselects) para filtrar el dataset en tiempo real.
2.  **Métricas Clave (KPIs):** Visualización de indicadores principales (ej: Promedio general, % de aprobación, total de alumnos en riesgo) que se actualicen según los filtros aplicados.
3.  **Visualización Dinámica:** Al menos dos gráficos que respondan a los filtros de la interfaz.
4.  **UX de Programador:** El código de la interfaz debe estar modularizado (uso de funciones) y manejar correctamente la carga de datos (caché).

**Nota**: Dado que Streamlit no se ejecuta directamente de forma interactiva dentro de las celdas de Colab de manera estándar, deben adjuntar el archivo .py en su repositorio de GitHub y, si lo desean, una captura de pantalla del dashboard funcionando en este cuaderno.

In [ ]:
# ⚠️ IMPORTANTE: El código completo de la interfaz de Streamlit (Hito 4) 
# ya se encuentra armado en el archivo app.py en esta misma carpeta.
# 
# Para ejecutarlo, abre una terminal en esta carpeta y corre el comando:
# streamlit run app.py


## Hito 5: Informe de Gestión y Propuesta de Mejora 🚀

### 1. Diagnóstico Académico
A través del modelado de datos, se detectó un patrón crítico: el bagaje socioeconómico (IMD) y educativo previo dictaminan fuertemente la probabilidad base de éxito, pero es el retraso en la entrega de las primeras evaluaciones (falta de engagement temporal) el principal indicador conductual que precede a un abandono ('Withdrawn').

### 2. Propuestas de Mejora (Justificadas en Datos)
* **Propuesta A: Sistema de Alerta Temprana Automatizado**
    * **Justificación:** La tasa de entregas tardías se dispara drásticamente en los perfiles de abandono. La universidad debe implementar un trigger automático: si un alumno entrega su primer TMA tarde, el sistema le enviará un correo de tutoría proactiva.
* **Propuesta B: Programa de Nivelación Focalizado**
    * **Justificación:** Los gráficos revelan una gran vulnerabilidad en estudiantes de las bandas IMD más bajas (0-20%) y nivel educativo bajo. Se propone una tutoría inicial obligatoria específicamente dirigida a estas cohortes durante el primer cuatrimestre para equilibrar la brecha estructural.

### 3. Conclusión Final
Implementar estrategias predictivas basadas en datos conductuales en tiempo real permitirá pasar de un modelo educativo reactivo a uno **proactivo**, mejorando la tasa de retención al detectar al estudiante en riesgo antes de que abandone.
